# run_dpwm notebook

This notebook embeds the full code from `run_dpwm.py` and runs `main()` directly.

In [ ]:
from pathlib import Path
import os

# Optional setup cell. Main code cell also sets project root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)
print(f"Working directory: {ROOT}")

In [ ]:
from pathlib import Path
import os

# Notebook: ensure project root (run big cell alone).
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
os.chdir(ROOT)

import numpy as np
import pandas as pd
from pathlib import Path
import argparse

from dpwm_model import DPWM
from calibrate_validate_dpwm import kge_2012
from calibration_io import BASIN_IDS, WARM_SPAN_DAYS, basin_hydrograph_title, load_preferred_calibration_df
N_DAYS_TOTAL = 4018
TRAIN_FRAC = 0.70
SPLIT_SEED = 42
INV_FLOW_OFFSET = 1.0
AREA_FIELD = "area_estre"


def _load_obs_cache(npz_path: Path) -> tuple[pd.DatetimeIndex, np.ndarray, list[str]]:
    data = np.load(npz_path, allow_pickle=True)
    obs_dates = pd.to_datetime(data["date"])
    q_obs = np.asarray(data["q_obs"], dtype=float)
    basin_ids = [str(x) for x in data["basin_ids"]]
    return obs_dates, q_obs, basin_ids


def _compute_comparable_metrics(
    q_sim: np.ndarray,
    q_obs_mmday: np.ndarray,
    pet: np.ndarray,
    basin_i: int,
) -> tuple[float, float, float, float]:
    """
    Metrics to match previous calibration-style setup:
      - kge2012
      - inverse flow term with c=1 -> 1/(Q+c)
      - valid days: finite Q_obs, finite PET, and day >= warm_span
      - random split (seed + basin_i), train_frac=0.70
      - returns: (KGE_Q_cal, KGE_invQ_cal, KGE_Q_val, KGE_invQ_val)
    """
    n = len(q_obs_mmday)
    idx_all = np.arange(n)
    valid_mask = np.isfinite(q_obs_mmday) & np.isfinite(pet) & (idx_all >= WARM_SPAN_DAYS)
    valid_idx = np.where(valid_mask)[0]
    if valid_idx.size < 2:
        return np.nan, np.nan, np.nan, np.nan

    rng = np.random.default_rng(SPLIT_SEED + basin_i)
    n_train = int(round(TRAIN_FRAC * valid_idx.size))
    n_train = max(1, min(n_train, valid_idx.size - 1))
    train_idx = rng.choice(valid_idx, size=n_train, replace=False)
    val_idx = np.setdiff1d(valid_idx, train_idx, assume_unique=False)

    def _kges(idxs: np.ndarray) -> tuple[float, float]:
        q_obs = q_obs_mmday[idxs]
        q_sim_i = q_sim[idxs]

        m = np.isfinite(q_obs) & np.isfinite(q_sim_i)
        if not np.any(m):
            return np.nan, np.nan
        q_obs = q_obs[m]
        q_sim_i = q_sim_i[m]

        kge_q = float(kge_2012(q_sim_i, q_obs))

        q_obs_pos = np.maximum(q_obs, 0.0)
        q_sim_pos = np.maximum(q_sim_i, 0.0)
        inv_obs = 1.0 / (q_obs_pos + INV_FLOW_OFFSET)
        inv_sim = 1.0 / (q_sim_pos + INV_FLOW_OFFSET)
        kge_invq = float(kge_2012(inv_sim, inv_obs))
        return kge_q, kge_invq

    kge_q_cal, kge_invq_cal = _kges(train_idx)
    kge_q_val, kge_invq_val = _kges(val_idx)
    return kge_q_cal, kge_invq_cal, kge_q_val, kge_invq_val


def main() -> None:
    """Run DPWM using calibration params/forcing and save hydrographs for 6 basins."""
    parser = argparse.ArgumentParser(
        description="Run DPWM for all basins and save discharge components; optionally generate hydrograph PNGs."
    )
    parser.add_argument(
        "--make_plots",
        action="store_true",
        help="Generate hydrograph PNG outputs (default: disabled for faster runs).",
    )
    args = parser.parse_known_args()[0]

    print("Loading forcing + calibrated parameters ...")
    rain_df = pd.read_csv("Results/forcing/rainplusmelt.csv")
    pet_df = pd.read_csv("Results/forcing/PET.csv")
    cal_df, cal_source = load_preferred_calibration_df()
    print(f"Using calibration file: {cal_source}")

    forcing_dates = pd.to_datetime(rain_df["date"])
    n_days = len(forcing_dates)

    # Simulate all requested basins with calibrated parameters from CSV
    sim_map: dict[str, np.ndarray] = {}
    qf_map: dict[str, np.ndarray] = {}
    qs_map: dict[str, np.ndarray] = {}
    rf_map: dict[str, np.ndarray] = {}
    rs_map: dict[str, np.ndarray] = {}
    et_map: dict[str, np.ndarray] = {}
    pet_map: dict[str, np.ndarray] = {}
    precip_map: dict[str, np.ndarray] = {}
    row_map: dict[str, pd.Series] = {}
    for i_basin, basin_id in enumerate(BASIN_IDS):
        row = cal_df.loc[cal_df["basin_id"] == basin_id]
        if row.empty:
            print(f"Skipping {basin_id}: missing in calibration_validation_results.csv")
            continue
        row = row.iloc[0]
        row_map[basin_id] = row

        if basin_id not in rain_df.columns or basin_id not in pet_df.columns:
            print(f"Skipping {basin_id}: basin not found in forcing CSV columns.")
            continue

        precip = rain_df[basin_id].to_numpy(dtype=float)
        pet = pet_df[basin_id].to_numpy(dtype=float)
        params = [
            float(row["alpha1"]),
            float(row["Sm"]),
            float(row["alpha2"]),
            float(row["Kf"]),
            float(row["Ks"]),
        ]

        print(f"Simulating basin {i_basin + 1}/{len(BASIN_IDS)} ({basin_id}) ...")
        model = DPWM(params, warm_span_days=WARM_SPAN_DAYS)
        flux_output, flux_internal, _ = model.simulate(precip, pet)
        sim_map[basin_id] = flux_output.Q
        qf_map[basin_id] = flux_internal.QF
        qs_map[basin_id] = flux_internal.QS
        rf_map[basin_id] = flux_internal.RF
        rs_map[basin_id] = flux_internal.RS
        et_map[basin_id] = flux_output.ET
        pet_map[basin_id] = pet
        precip_map[basin_id] = precip

    # Save discharge components explicitly for downstream workflows.
    components_dir = Path("Results/discharge_components")
    components_dir.mkdir(parents=True, exist_ok=True)
    qf_wide = pd.DataFrame({"date": forcing_dates})
    qs_wide = pd.DataFrame({"date": forcing_dates})
    q_wide = pd.DataFrame({"date": forcing_dates})

    for basin_id in BASIN_IDS:
        if basin_id not in sim_map:
            continue
        comp_df = pd.DataFrame(
            {
                "date": forcing_dates,
                "precip": precip_map[basin_id],
                "pet": pet_map[basin_id],
                "Q_total": sim_map[basin_id],
                "Q_fast": qf_map[basin_id],
                "Q_slow": qs_map[basin_id],
                "RF": rf_map[basin_id],
                "RS": rs_map[basin_id],
                "ET": et_map[basin_id],
            }
        )
        comp_out = components_dir / f"discharge_components_{basin_id}.csv"
        comp_df.to_csv(comp_out, index=False)
        print(f"Saved discharge components: {comp_out}")

        q_wide[basin_id] = sim_map[basin_id]
        qf_wide[basin_id] = qf_map[basin_id]
        qs_wide[basin_id] = qs_map[basin_id]

    q_wide_path = components_dir / "Q_total_all_basins.csv"
    qf_wide_path = components_dir / "Q_fast_all_basins.csv"
    qs_wide_path = components_dir / "Q_slow_all_basins.csv"
    q_wide.to_csv(q_wide_path, index=False)
    qf_wide.to_csv(qf_wide_path, index=False)
    qs_wide.to_csv(qs_wide_path, index=False)
    print(f"Saved combined total discharge: {q_wide_path}")
    print(f"Saved combined fast discharge: {qf_wide_path}")
    print(f"Saved combined slow discharge: {qs_wide_path}")

    if not args.make_plots:
        print("Plotting disabled. Use --make_plots to also generate hydrograph PNG files.")
        return

    try:
        from plots import plot_hydrograph_obs_sim
    except ImportError as exc:
        raise ImportError(
            "Hydrograph plotting needs plots.py, which is not in this repository. "
            "Run without --make_plots, or use run_hydro_plots.py / plot_future_hydrograph.py."
        ) from exc

    # Prefer the exact calibration-aligned Q cache (same forcing grid, same basin order).
    obs_aligned_path = Path(
        "Results/cache_obs/Q_obs_CH000127_DESN1585_FR002785_GB000215_SE000197_ES000700.npz"
    )
    if obs_aligned_path.exists():
        obs_aligned = np.load(obs_aligned_path, allow_pickle=True)["q_obs_aligned"]
        if obs_aligned.shape != (n_days, len(BASIN_IDS)):
            raise ValueError(
                f"Aligned obs shape {obs_aligned.shape} does not match forcing grid "
                f"({n_days}, {len(BASIN_IDS)})."
            )
        use_aligned_obs = True
    else:
        use_aligned_obs = False
        # Fallback to dated 6-basin cache.
        obs_npz_path = Path("Results/cache_obs/Q_obs_6basins.npz")
        if not obs_npz_path.exists():
            raise FileNotFoundError(
                "Missing observed discharge cache. Expected either "
                f"{obs_aligned_path} or {obs_npz_path}."
            )
        obs_dates, q_obs_all, obs_basin_ids = _load_obs_cache(obs_npz_path)
        obs_index_map = {bid: i for i, bid in enumerate(obs_basin_ids)}

    # Save hydrographs here
    out_dir = Path("Results/hydrographs")
    out_dir.mkdir(parents=True, exist_ok=True)
    zoom_dir = Path("Results/hydrographs_zoom_2006_2009")
    zoom_dir.mkdir(parents=True, exist_ok=True)
    metrics_rows = []

    for i_basin, basin_id in enumerate(BASIN_IDS):
        if basin_id not in sim_map:
            continue
        row = row_map[basin_id]
        sim_df = pd.DataFrame(
            {
                "date": forcing_dates,
                "precip": precip_map[basin_id],
                "pet": pet_map[basin_id],
                "q_sim": sim_map[basin_id],
            }
        )
        if use_aligned_obs:
            sim_df["q_obs"] = obs_aligned[:, i_basin]
            merged = sim_df
        else:
            if basin_id not in obs_index_map:
                print(f"Skipping {basin_id}: no observed discharge in cache.")
                continue
            obs_df = pd.DataFrame(
                {
                    "date": obs_dates,
                    "q_obs": q_obs_all[:, obs_index_map[basin_id]],
                }
            )
            merged = sim_df.merge(obs_df, on="date", how="inner").dropna(subset=["q_obs"])
            if merged.empty:
                print(f"Skipping {basin_id}: no overlapping obs/sim dates.")
                continue

        # Convert observed Q from m3/s to mm/day (same convention as calibration script)
        area_km2 = float(row["area_km2"]) if "area_km2" in row.index else np.nan
        if not np.isfinite(area_km2) or area_km2 <= 0.0:
            print(f"Skipping {basin_id}: invalid area for mm/day conversion ({area_km2}).")
            continue
        merged["q_obs_mmday"] = merged["q_obs"] * 86.4 / area_km2

        # Match exact calibration window from calibration CSV row
        start_idx = int(row["window_start_idx"])
        window_days = int(row["window_days"])
        if start_idx < 0 or start_idx >= len(forcing_dates):
            print(f"Skipping {basin_id}: invalid window_start_idx={start_idx}.")
            continue
        end_idx = min(start_idx + window_days, len(forcing_dates))
        win_start_date = forcing_dates.iloc[start_idx]
        win_end_date = forcing_dates.iloc[end_idx - 1]
        metric_df = merged[(merged["date"] >= win_start_date) & (merged["date"] <= win_end_date)].copy()
        if metric_df.empty:
            print(f"Skipping {basin_id}: no data in hydro window.")
            continue

        kge_q_cal, kge_invq_cal, kge_q_val, kge_invq_val = _compute_comparable_metrics(
            q_sim=metric_df["q_sim"].to_numpy(dtype=float),
            q_obs_mmday=metric_df["q_obs_mmday"].to_numpy(dtype=float),
            pet=metric_df["pet"].to_numpy(dtype=float),
            basin_i=i_basin,
        )

        metrics_text = (
            f"KGE(Q)_cal = {kge_q_cal:.3f}\n"
            f"KGE(1/(Q+1))_cal = {kge_invq_cal:.3f}\n"
            f"KGE(Q)_val = {kge_q_val:.3f}\n"
            f"KGE(1/(Q+1))_val = {kge_invq_val:.3f}"
        )
        metrics_rows.append(
            {
                "basin_id": basin_id,
                "KGE_Q_cal": kge_q_cal,
                "KGE_invQ_cal": kge_invq_cal,
                "KGE_Q_val": kge_q_val,
                "KGE_invQ_val": kge_invq_val,
            }
        )

        out_png = out_dir / f"hydrograph_{basin_id}.png"
        plot_hydrograph_obs_sim(
            basin_id=basin_id,
            dates=merged["date"],
            precip=merged["precip"].to_numpy(dtype=float),
            q_obs=merged["q_obs_mmday"].to_numpy(dtype=float),
            q_sim=merged["q_sim"].to_numpy(dtype=float),
            out_path=str(out_png),
            show=False,
            metrics_text=metrics_text,
        )
        print(f"Saved hydrograph: {out_png}")

        # Save zoomed hydrograph (2006-2009) with dashed simulated Q
        zoom_df = merged[
            (merged["date"] >= pd.Timestamp("2006-01-01"))
            & (merged["date"] <= pd.Timestamp("2009-12-31"))
        ].copy()
        if not zoom_df.empty:
            out_zoom_png = zoom_dir / f"hydrograph_{basin_id}_2006_2009.png"
            plot_hydrograph_obs_sim(
                basin_id=basin_id,
                dates=zoom_df["date"],
                precip=zoom_df["precip"].to_numpy(dtype=float),
                q_obs=zoom_df["q_obs_mmday"].to_numpy(dtype=float),
                q_sim=zoom_df["q_sim"].to_numpy(dtype=float),
                out_path=str(out_zoom_png),
                show=False,
                metrics_text=None,
                q_sim_linestyle="-",
                q_obs_markers_only=True,
                q_obs_marker="x",
                q_obs_markersize=2.0,
                top_title=basin_hydrograph_title(basin_id),
                hydrograph_title="",
                overlay_precip=True,
                show_metrics=False,
            )
            print(f"Saved zoom hydrograph: {out_zoom_png}")

    if metrics_rows:
        metrics_df = pd.DataFrame(metrics_rows)
        metrics_csv = out_dir / "hydrograph_metrics.csv"
        metrics_df.to_csv(metrics_csv, index=False)
        print(f"Saved metrics table: {metrics_csv}")


# Execute the script entry point
main()
